# 05 - Risk Map Generation
Generates an interactive Folium wildfire risk map for the most recent demo week, plus a sample fire-spread simulation from the highest-risk cell.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "04_Source_Code"))
import config
from utils import get_logger
logger = get_logger("notebook")
print("Project root:", config.PROJECT_ROOT)
print("Study region:", config.STUDY_REGION["name"])


In [ ]:
from data_loader import load_dataset
from train_models import train_all_models
from evaluate_models import evaluate_all_models, best_model_name
from risk_mapping import build_risk_table, generate_risk_map_html

df = load_dataset(source="demo")
result = train_all_models(df)
comparison = evaluate_all_models(result)
best = best_model_name(comparison)
best_bundle = result["models"][best]

latest_date = sorted(df["date"].unique())[-1]
latest_cells = df[df["date"] == latest_date].reset_index(drop=True)
X_latest = (best_bundle["scaler"].transform(latest_cells[config.FEATURE_COLUMNS])
            if best_bundle.get("uses_scaler") else latest_cells[config.FEATURE_COLUMNS])
proba = best_bundle["model"].predict_proba(X_latest)[:,1]
risk_df = build_risk_table(latest_cells, proba)
risk_df["risk_category"].value_counts()


In [ ]:
map_path = generate_risk_map_html(risk_df, out_name="notebook_risk_map.html")
print("Saved:", map_path)


## Sample fire-spread simulation from the highest-risk cell

In [ ]:
import numpy as np
from generate_demo_data import build_spatial_grid, _assign_static_terrain_and_landcover
from fire_spread import run_fire_spread_simulation, generate_spread_map_html

rng = np.random.default_rng(config.RANDOM_SEED)
grid = build_spatial_grid()
grid = _assign_static_terrain_and_landcover(grid, rng)
grid = grid.merge(latest_cells[["grid_id","ndmi"]], on="grid_id", how="left")
grid["ndmi"] = grid["ndmi"].fillna(0.15)

ignition_id = risk_df.sort_values("risk_probability", ascending=False).iloc[0]["grid_id"]
sim = run_fire_spread_simulation(grid, ignition_grid_id=ignition_id, wind_speed=25,
                                  wind_direction=90, duration_minutes=60)
print("Ignition:", ignition_id, "| Affected cells:", len(sim["affected_cells"]),
      "| Area (km2):", sim["affected_area_km2"])
generate_spread_map_html(sim, out_name="notebook_spread_map.html")


## Disclaimer
Risk categories use prototype thresholds (Low/Moderate/High/Extreme at 0.25/0.50/0.75) that require calibration and validation before any real-world or scientific use. The fire-spread simulation is a simplified, explainable cellular-automata approximation, not a physically exact fire-behaviour model.